### CONFIGURACION GENERAL
Definimos la estructura moderna. Ya no usamos DBFS, ahora todo vive en Volumes.

In [0]:
catalog="main"
schema = "school"
volume="raw_data"

# Modern base path (Volumes)
base_path = f"/Volumes/{catalog}/{schema}/{volume}"

# Subfolders
students_path = f"{base_path}/students-json"
courses_path = f"{base_path}/courses-csv"

print("Base path:", base_path)
print("Students path:", students_path)
print("Courses path:", courses_path)

Base path: /Volumes/main/school/raw_data
Students path: /Volumes/main/school/raw_data/students-json
Courses path: /Volumes/main/school/raw_data/courses-csv


### Create structure (Unity Catalog)
This completely replaces DBFS: now we have catalog, schema, and volume with real governance.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

print("Structure created successfully")

Structure created successfully


### SOURCE (S3)
S3 is the data source. This is where the files originally reside.

In [0]:
s3_students = "s3://dalhussein-books/DEA-Book/datasets/school/v1/students-json"
s3_courses = "s3://dalhussein-books/DEA-Book/datasets/school/v1/courses-csv"

### Ingestion (copy to Volumes)
We bring the data from S3 to Volumes so we can manage it within the lakehouse.

In [0]:
dbutils.fs.cp(s3_students, students_path, recurse=True)
dbutils.fs.cp(s3_courses, courses_path, recurse=True)

print("Data copied to Volumes")

Data copied to Volumes


In [0]:
files = dbutils.fs.ls(students_path)
display(files)

path,name,size,modificationTime
dbfs:/Volumes/main/school/raw_data/students-json/export_001.json,export_001.json,82932,1775959573000
dbfs:/Volumes/main/school/raw_data/students-json/export_002.json,export_002.json,83399,1775959573000
dbfs:/Volumes/main/school/raw_data/students-json/export_003.json,export_003.json,83232,1775959574000
dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,export_004.json,83489,1775959574000
dbfs:/Volumes/main/school/raw_data/students-json/export_005.json,export_005.json,83208,1775959575000
dbfs:/Volumes/main/school/raw_data/students-json/export_006.json,export_006.json,55562,1775959575000


### Querying Data Files (Exploring)
There are no tables here. Spark is reading files directly as if they were temporary tables.


-- 1️ ASTERISK (*) - All files
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/*`

-- 2️ ASTERISK WITH EXTENSION - JSONs only
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/*.json`

-- 3️ MULTIPLE LEVELS (**)
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/**`

-- 4️ PATTERN WITH NUMBERS
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/export_*.json`

-- 5️ SPECIFIC PATTERN (export_001, export_002, etc)
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/export_00[1-6].json`

-- 6️ NUMBER RANGE (001 to 005)
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/export_00[1-5].json`

-- 7️ PATTERN WITH QUESTION (one character)
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/export_00?.json`

-- 8️ MULTIPLE EXTENSIONS (JSON or CSV)
SELECT * FROM json.`/Volumes/main/school/raw_data/students-json/*{.json,.csv}`

-- 9️ SPECIFIC FOLDERS
SELECT * FROM json.`/Volumes/main/school/raw_data/*/export_*.json`

-- 10 ALL (nested folders and files)
SELECT * FROM json.`/Volumes/main/school/raw_data/**/*.json`

In [0]:
%sql
-- QUERY ON FILES (JSON)
SELECT * 
FROM json.`/Volumes/main/school/raw_data/students-json/*`

email,gpa,profile,student_id,updated
glenard3v@miitbeian.gov.cn,3.84,"{""first_name"":""Gregoor"",""last_name"":""Lenard"",""gender"":""Male"",""address"":{""street"":""0 Superior Park"",""city"":""Trelleborg"",""country"":""Sweden""}}",S00901,2021-12-14T23:15:43.375Z
null,2.56,"{""first_name"":""Pearla"",""last_name"":""Lengthorn"",""gender"":""Female"",""address"":{""street"":""75 Dottie Way"",""city"":""Zengtian"",""country"":""United Kingdom""}}",S00902,2021-12-14T23:15:43.375Z
plequeux9p@delicious.com,1.34,"{""first_name"":""Parker"",""last_name"":""Lequeux"",""gender"":""Male"",""address"":{""street"":""70 Badeau Lane"",""city"":""Jastrzębia"",""country"":""Poland""}}",S00903,2021-12-14T23:15:43.375Z
dlewcockda@pen.io,3.74,"{""first_name"":""Dane"",""last_name"":""Lewcock"",""gender"":""Genderqueer"",""address"":{""street"":""6 Sloan Lane"",""city"":""Yajiwa"",""country"":""Nigeria""}}",S00904,2021-12-14T23:15:43.375Z
uleynaghdt@admin.ch,2.04,"{""first_name"":""Ulrikaumeko"",""last_name"":""Leynagh"",""gender"":""Female"",""address"":{""street"":""66975 Division Avenue"",""city"":""New Panamao"",""country"":""Philippines""}}",S00905,2021-12-14T23:15:43.375Z
blibbie6n@google.es,3.84,"{""first_name"":""Bea"",""last_name"":""Libbie"",""gender"":""Female"",""address"":{""street"":""9 Ruskin Junction"",""city"":""Wielichowo"",""country"":""Poland""}}",S00906,2021-12-14T23:15:43.375Z
null,2.12,"{""first_name"":""Marcello"",""last_name"":""Liddy"",""gender"":""Non-binary"",""address"":{""street"":""8157 Sunnyside Hill"",""city"":""Moog"",""country"":""Philippines""}}",S00907,2021-12-14T23:15:43.375Z
null,2.79,"{""first_name"":""Waldo"",""last_name"":""Lilford"",""gender"":""Male"",""address"":{""street"":""16 Fairfield Circle"",""city"":""Kwidzyn"",""country"":""Poland""}}",S00908,2021-12-14T23:15:43.375Z
jlill14@census.gov,2.81,"{""first_name"":""Janela"",""last_name"":""Lill"",""gender"":""Female"",""address"":{""street"":""4 Pawling Way"",""city"":""Tambaú"",""country"":""Brazil""}}",S00909,2021-12-14T23:15:43.375Z
null,2.66,"{""first_name"":""Bevin"",""last_name"":""Lille"",""gender"":""Male"",""address"":{""street"":""89402 Katie Way"",""city"":""Zelenograd"",""country"":""Russia""}}",S00910,2021-12-14T23:15:43.375Z


### Metadata


In [0]:
%sql
SELECT 
  *,
  _metadata.file_path as source_file,
  _metadata.file_modification_time as file_mod_time
FROM json.`/Volumes/main/school/raw_data/students-json/*`

email,gpa,profile,student_id,updated,source_file,file_mod_time
glenard3v@miitbeian.gov.cn,3.84,"{""first_name"":""Gregoor"",""last_name"":""Lenard"",""gender"":""Male"",""address"":{""street"":""0 Superior Park"",""city"":""Trelleborg"",""country"":""Sweden""}}",S00901,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
null,2.56,"{""first_name"":""Pearla"",""last_name"":""Lengthorn"",""gender"":""Female"",""address"":{""street"":""75 Dottie Way"",""city"":""Zengtian"",""country"":""United Kingdom""}}",S00902,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
plequeux9p@delicious.com,1.34,"{""first_name"":""Parker"",""last_name"":""Lequeux"",""gender"":""Male"",""address"":{""street"":""70 Badeau Lane"",""city"":""Jastrzębia"",""country"":""Poland""}}",S00903,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
dlewcockda@pen.io,3.74,"{""first_name"":""Dane"",""last_name"":""Lewcock"",""gender"":""Genderqueer"",""address"":{""street"":""6 Sloan Lane"",""city"":""Yajiwa"",""country"":""Nigeria""}}",S00904,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
uleynaghdt@admin.ch,2.04,"{""first_name"":""Ulrikaumeko"",""last_name"":""Leynagh"",""gender"":""Female"",""address"":{""street"":""66975 Division Avenue"",""city"":""New Panamao"",""country"":""Philippines""}}",S00905,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
blibbie6n@google.es,3.84,"{""first_name"":""Bea"",""last_name"":""Libbie"",""gender"":""Female"",""address"":{""street"":""9 Ruskin Junction"",""city"":""Wielichowo"",""country"":""Poland""}}",S00906,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
null,2.12,"{""first_name"":""Marcello"",""last_name"":""Liddy"",""gender"":""Non-binary"",""address"":{""street"":""8157 Sunnyside Hill"",""city"":""Moog"",""country"":""Philippines""}}",S00907,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
null,2.79,"{""first_name"":""Waldo"",""last_name"":""Lilford"",""gender"":""Male"",""address"":{""street"":""16 Fairfield Circle"",""city"":""Kwidzyn"",""country"":""Poland""}}",S00908,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
jlill14@census.gov,2.81,"{""first_name"":""Janela"",""last_name"":""Lill"",""gender"":""Female"",""address"":{""street"":""4 Pawling Way"",""city"":""Tambaú"",""country"":""Brazil""}}",S00909,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z
null,2.66,"{""first_name"":""Bevin"",""last_name"":""Lille"",""gender"":""Male"",""address"":{""street"":""89402 Katie Way"",""city"":""Zelenograd"",""country"":""Russia""}}",S00910,2021-12-14T23:15:43.375Z,dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,2026-04-12T02:06:14.000Z


In [0]:
%sql
-- QUERY ON FILES (CSV)
SELECT * 
FROM csv.`/Volumes/main/school/raw_data/courses-csv/*`

_c0
course_id;title;instructor;category;price
C01;Data Structures and Algorithms;Tracy N.;Computer Science;49
C02;JavaScript Design Patterns;Ali M.;Computer Science;28
C03;Neural Network;Adam R.;Computer Science;35
course_id;title;instructor;category;price
C04;Robot Dynamics and Control;Mark G.;Computer Science;20
C05;Python Programming;Luciano C.;Computer Science;47
C06;Deep Learning;François R.;Computer Science;22
course_id;title;instructor;category;price
C07;Machine Learning;Andriy R.;Computer Science;33


### Create Delta Tables (Bronze → Silver)
Here we go from raw files to Delta tables. We now have transactions, control, and performance.

In [0]:
%sql
-- CREATE STUDENTS TABLE
CREATE OR REPLACE TABLE main.school.students
AS
SELECT * 
FROM json.`/Volumes/main/school/raw_data/students-json/*`

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- CREATE COURSES TABLE
CREATE OR REPLACE TABLE main.school.courses_csv
AS
SELECT * 
FROM csv.`/Volumes/main/school/raw_data/courses-csv/*`

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM main.school.courses_csv

_c0
course_id;title;instructor;category;price
C10;Database Design Solutions;Julia S.;Computer Science;44
C11;Business Intelligence;Tiffany M.;Computer Science;38
C12;Big Data;Bernard M.;Computer Science;30
course_id;title;instructor;category;price
C01;Data Structures and Algorithms;Tracy N.;Computer Science;49
C02;JavaScript Design Patterns;Ali M.;Computer Science;28
C03;Neural Network;Adam R.;Computer Science;35
course_id;title;instructor;category;price
C07;Machine Learning;Andriy R.;Computer Science;33


In [0]:
%sql
CREATE OR REPLACE TABLE main.school.courses AS 
SELECT * 
FROM read_files(
  '/Volumes/main/school/raw_data/courses-csv/',
  format => 'csv',
  header => true,
  delimiter => ';'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from main.school.courses

course_id,title,instructor,category,price,_rescued_data
C10,Database Design Solutions,Julia S.,Computer Science,44,null
C11,Business Intelligence,Tiffany M.,Computer Science,38,null
C12,Big Data,Bernard M.,Computer Science,30,null
C01,Data Structures and Algorithms,Tracy N.,Computer Science,49,null
C02,JavaScript Design Patterns,Ali M.,Computer Science,28,null
C03,Neural Network,Adam R.,Computer Science,35,null
C07,Machine Learning,Andriy R.,Computer Science,33,null
C08,Quantum Computing,Chris N.,Computer Science,41,null
C09,Advanced Data Structures,Pierre B.,Computer Science,24,null
C04,Robot Dynamics and Control,Mark G.,Computer Science,20,null


In [0]:
%sql
CREATE OR REPLACE TABLE main.school.enrollments AS
SELECT
  s.student_id,
  c.course_id AS course_id,
  CURRENT_TIMESTAMP() AS timestamp
FROM main.school.students s
CROSS JOIN main.school.courses c
LIMIT 100

num_affected_rows,num_inserted_rows


### Real queries (production work)
Once in Delta, you can do joins, inserts, merges... everything from production.

In [0]:
%sql
SELECT 
  s.email,
  s.gpa,
  c.title
FROM main.school.students s
JOIN main.school.enrollments e
  ON s.student_id = e.student_id
JOIN main.school.courses c
  ON e.course_id = c.course_id

email,gpa,title
dabby2y@japanpost.jp,1.48,Database Design Solutions
eabbysc1@github.com,3.02,Business Intelligence
rabelovd1@wikispaces.com,3.31,Big Data
rabels9g@behance.net,1.89,Data Structures and Algorithms
sabendrothin@cargocollective.com,3.55,JavaScript Design Patterns
null,2.9,Neural Network
sabrahmson3h@blinklist.com,2.96,Machine Learning
dacheson2h@mapy.cz,1.2,Quantum Computing
fackwoodji@gravatar.com,1.96,Advanced Data Structures
null,1.39,Robot Dynamics and Control


### Images

In [0]:
%sql
SELECT * FROM binaryFile.`/Volumes/main/school/raw_data/images/cats.jpg`

path,modificationTime,length,content
dbfs:/Volumes/main/school/raw_data/images/cats.jpg,2026-04-12T02:03:22.000Z,84510,...


### CLEAN UP

In [0]:
def clean_up():
    print("Deleting volume...")
    dbutils.fs.rm(base_path, True)
    
    print("Deleting schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
    
    print("Done")

In [0]:
clean_up()

Deleting volume...
Deleting schema...
Done
